# 10. Experiment 1 — Word-context representation

In [4]:
import numpy as np
import pandas as pd
from collections import Counter


corpus = [
    "the doctor treated the patient in the hospital",
    "the physician treated the patient with care",
    "the doctor works in the hospital",
    "the physician works in the hospital",
    "a nurse helps the doctor and patient",
    "monkeys eat sweet yellow bananas",
    "monkeys like bananas and fruit"
]

tokens_list = [doc.lower().split() for doc in corpus]

all_tokens = [token for doc in tokens_list for token in doc]
vocab = sorted(list(set(all_tokens)))
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

def build_cooccurrence_matrix(tokens_list, vocab_size, word2idx, window_size=1):
    matrix = np.zeros((vocab_size, vocab_size), dtype=np.int32)
    for tokens in tokens_list:
        for i, target_word in enumerate(tokens):
            target_idx = word2idx[target_word]
            # Lấy các từ nằm trong khoảng [i - k, i + k] (trừ chính từ đó)
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)
            for j in range(start, end):
                if i != j:
                    context_word = tokens[j]
                    context_idx = word2idx[context_word]
                    matrix[target_idx, context_idx] += 1
    return matrix

def cosine_sim(v1, v2):
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm1 * norm2)

windows = [1, 2, 5]
results = []

print(f"Vocabulary size (|V|): {vocab_size}")
print(f"Matrix shape: {vocab_size} x {vocab_size}\n")

for k in windows:
    mat = build_cooccurrence_matrix(tokens_list, vocab_size, word2idx, window_size=k)
    nnz = np.count_nonzero(mat) # Số lượng ô khác 0
    
    vec_doctor = mat[word2idx["doctor"]]
    vec_physician = mat[word2idx["physician"]]
    vec_hospital = mat[word2idx["hospital"]]
    
    sim_doc_phy = cosine_sim(vec_doctor, vec_physician)
    sim_doc_hos = cosine_sim(vec_doctor, vec_hospital)
    
    results.append({
        "Window (k)": k,
        "Matrix Size": f"{vocab_size}x{vocab_size}",
        "Non-zero Values": nnz,
        "Sparsity (%)": round((1 - nnz / (vocab_size**2)) * 100, 2),
        "sim(doctor, physician)": round(sim_doc_phy, 4),
        "sim(doctor, hospital)": round(sim_doc_hos, 4)
    })

df_exp1 = pd.DataFrame(results)
print(df_exp1.to_string(index=False))

Vocabulary size (|V|): 21
Matrix shape: 21 x 21

 Window (k) Matrix Size  Non-zero Values  Sparsity (%)  sim(doctor, physician)  sim(doctor, hospital)
          1       21x21               54         87.76                  0.9428                 0.8660
          2       21x21               90         79.59                  0.9232                 0.7538
          5       21x21              141         68.03                  0.9051                 0.9040


# 17.Experiment 2 — Word2Vec

Corpus: Tập dữ liệu C4 (30,000 văn bản tiếng Anh thô) sau khi qua bước làm sạch (chuyển chữ thường, xóa URL/HTML/ký tự đặc biệt) và tách từ (tokenization).
Vocabulary: Tập từ vựng thu được gồm ~85,000 từ duy nhất
Embedding Dimension (vector_size): 100
Context Window (window): 5
Min Count (min_count): 2
Epochs (epochs): 10
Training Objective (sg): Skip-gram with Negative Sampling (SGNS) (sg=1

In [5]:
import json
import re
import pandas as pd
from gensim.models import Word2Vec

file_path = "c4-train.00000-of-01024-30K.json"

c4_data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        # Lấy trường 'text' từ mỗi bản ghi JSON
        c4_data.append(data["text"])

print(f"Đã load thành công {len(c4_data)} bản ghi C4.")

def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+|<.*?>', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    return tokens

corpus = [clean_and_tokenize(doc) for doc in c4_data]

w2v_model = Word2Vec(
    sentences=corpus,        
    vector_size=100,         
    window=5,                
    min_count=2,             
    epochs=10,               
    sg=1,                    
    workers=4                
)

vocab_size = len(w2v_model.wv)
print(f"Tổng số từ trong Vocabulary (min_count>=2): {vocab_size}")

Đã load thành công 30000 bản ghi C4.
Tổng số từ trong Vocabulary (min_count>=2): 102354


# 18. Inspect Embedding

In [6]:
target_words = [
    "doctor", "hospital", "patient", "disease",
    "computer", "football", "banana"
]

results = {}
for word in target_words:
    if word in w2v_model.wv:
        similar_words = w2v_model.wv.most_similar(word, topn=5)
        results[word] = similar_words
        print(f"=== Top-5 từ gần nhất với '{word}' ===")
        for sim_word, score in similar_words:
            print(f"  {sim_word}: {score:.4f}")
        print()
    else:
        print(f"Từ '{word}' không có trong từ vựng (out of vocabulary).\n")

=== Top-5 từ gần nhất với 'doctor' ===
  obstetrician: 0.7642
  dentist: 0.7612
  physician: 0.7353
  pharmacist: 0.7285
  optometrist: 0.7284

=== Top-5 từ gần nhất với 'hospital' ===
  upmc: 0.7689
  icu: 0.7298
  hospitals: 0.7175
  clinic: 0.7082
  crittenton: 0.7079

=== Top-5 từ gần nhất với 'patient' ===
  patients: 0.7949
  postsurgical: 0.7277
  antenatal: 0.7253
  implantable: 0.7031
  subspecialty: 0.6988

=== Top-5 từ gần nhất với 'disease' ===
  diseases: 0.8267
  diabetes: 0.8155
  chronic: 0.8032
  infection: 0.8028
  autoimmune: 0.7917

=== Top-5 từ gần nhất với 'computer' ===
  uwowned: 0.7685
  laptop: 0.7371
  desktop: 0.7235
  computers: 0.7137
  software: 0.7109

=== Top-5 từ gần nhất với 'football' ===
  replica: 0.7942
  shirt: 0.7517
  hockey: 0.7367
  shirts: 0.7346
  soccer: 0.7307

=== Top-5 từ gần nhất với 'banana' ===
  tortilla: 0.7356
  pumpkin: 0.7244
  cranberries: 0.7197
  sprinkled: 0.7169
  marinara: 0.7159



# 19. Experiment 3 — Context window

In [7]:
from gensim.models import Word2Vec
import pandas as pd

word_pairs = [
    ("doctor", "physician"),
    ("doctor", "hospital"),
    ("cat", "dog")
]

print("Đang huấn luyện Model (window=2)...")
model_w2 = Word2Vec(corpus, vector_size=100, window=2, min_count=2, epochs=10, sg=1, workers=4)

print("Đang huấn luyện Model (window=5)...")
model_w5 = Word2Vec(corpus, vector_size=100, window=5, min_count=2, epochs=10, sg=1, workers=4)

print("Đang huấn luyện Model (window=10)...")
model_w10 = Word2Vec(corpus, vector_size=100, window=10, min_count=2, epochs=10, sg=1, workers=4)

data = []
for w1, w2 in word_pairs:
    sim_w2 = model_w2.wv.similarity(w1, w2) if (w1 in model_w2.wv and w2 in model_w2.wv) else None
    sim_w5 = model_w5.wv.similarity(w1, w2) if (w1 in model_w5.wv and w2 in model_w5.wv) else None
    sim_w10 = model_w10.wv.similarity(w1, w2) if (w1 in model_w10.wv and w2 in model_w10.wv) else None
    
    data.append({
        "Word pair": f"{w1} – {w2}",
        "Window 2": round(sim_w2, 4),
        "Window 5": round(sim_w5, 4),
        "Window 10": round(sim_w10, 4)
    })

df_results = pd.DataFrame(data)
print("\n=== BẢNG SO SÁNH COSINE SIMILARITY ===")
print(df_results.to_string(index=False))

Đang huấn luyện Model (window=2)...
Đang huấn luyện Model (window=5)...
Đang huấn luyện Model (window=10)...

=== BẢNG SO SÁNH COSINE SIMILARITY ===
         Word pair  Window 2  Window 5  Window 10
doctor – physician    0.7501    0.7514     0.7425
 doctor – hospital    0.5116    0.5680     0.6080
         cat – dog    0.6686    0.7030     0.7192


# 20. Experiment 4 — Embedding dimension

In [8]:
import time
import os
import numpy as np
import pandas as pd
from gensim.models import Word2Vec

dimensions = [50, 100, 300]
results = []

sim_pair = ("doctor", "physician")
analogy_words = (['germany', 'paris'], ['france']) # germany - france + paris = berlin

for d in dimensions:
    print(f"\n--- Đang huấn luyện với dimension = {d} ---")
    
    start_time = time.time()
    model = Word2Vec(
        sentences=corpus,
        vector_size=d,
        window=5,
        min_count=2,
        epochs=10,
        sg=1,
        workers=4
    )
    train_time = round(time.time() - start_time, 2)
    model_filename = f"w2v_dim_{d}.model"
    model.save(model_filename)
    model_size_mb = round(os.path.getsize(model_filename) / (1024 * 1024), 2)
    
    sim_score = round(model.wv.similarity(sim_pair[0], sim_pair[1]), 4) if all(w in model.wv for w in sim_pair) else None
    try:
        analogy_res = model.wv.most_similar(positive=analogy_words[0], negative=analogy_words[1], topn=1)[0][0]
    except KeyError:
        analogy_res = "N/A"
        
    
    doc1 = ["doctor", "treats", "patient", "in", "hospital"]
    doc2 = ["physician", "gives", "medical", "care", "to", "sick", "people"]
    
    def get_doc_vector(doc, mod, dim):
        vecs = [mod.wv[w] for w in doc if w in mod.wv]
        return np.mean(vecs, axis=0) if len(vecs) > 0 else np.zeros(dim)
    
    v1 = get_doc_vector(doc1, model, d)
    v2 = get_doc_vector(doc2, model, d)
  
    downstream_score = round(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)), 4)
    
    results.append({
        "Dimension": d,
        "Training Time (s)": train_time,
        "Model Size (MB)": model_size_mb,
        f"Sim ({sim_pair[0]}-{sim_pair[1]})": sim_score,
        "Analogy Top-1": analogy_res,
        "Downstream Task Sim": downstream_score
    })

df_exp4 = pd.DataFrame(results)
print("\n=== BẢNG TỔNG HỢP KẾT QUẢ EXPERIMENT 4 ===")
print(df_exp4.to_string(index=False))


csv_rows = []

for item in results:
    d = item["Dimension"]
    
    # 1. Metric: Training Time
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "N/A",
        "metric_name": "training_time_sec",
        "metric_value": item["Training Time (s)"]
    })
    
    # 2. Metric: Model Size
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "N/A",
        "metric_name": "model_size_mb",
        "metric_value": item["Model Size (MB)"]
    })
    
    # 3. Metric: Cosine Similarity (doctor - physician)
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": f"{sim_pair[0]}-{sim_pair[1]}",
        "metric_name": "cosine_similarity",
        "metric_value": item[f"Sim ({sim_pair[0]}-{sim_pair[1]})"]
    })
    
    # 4. Metric: Downstream Task Similarity
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "doc1-doc2",
        "metric_name": "downstream_doc_sim",
        "metric_value": item["Downstream Task Sim"]
    })

# Chuyển thành DataFrame
df_csv_exp4 = pd.DataFrame(csv_rows)

# Kiểm tra nếu file results.csv đã tồn tại thì nối thêm (append), chưa có thì tạo mới
csv_file = "results.csv"
if os.path.exists(csv_file):
    # Đọc dữ liệu cũ và ghi đè/nối dòng mới
    df_existing = pd.read_csv(csv_file)
    # Loại bỏ các dòng cũ của exp4_dimension nếu có (tránh bị lặp trùng khi rerun cell)
    df_existing = df_existing[df_existing["experiment"] != "exp4_dimension"]
    df_final = pd.concat([df_existing, df_csv_exp4], ignore_index=True)
else:
    df_final = df_csv_exp4

# Xuất ra file CSV
df_final.to_csv(csv_file, index=False, encoding="utf-8")
print(f"\n[OK] Đã lưu thành công kết quả Experiment 4 vào '{csv_file}'!")


--- Đang huấn luyện với dimension = 50 ---

--- Đang huấn luyện với dimension = 100 ---

--- Đang huấn luyện với dimension = 300 ---

=== BẢNG TỔNG HỢP KẾT QUẢ EXPERIMENT 4 ===
 Dimension  Training Time (s)  Model Size (MB)  Sim (doctor-physician) Analogy Top-1  Downstream Task Sim
        50             239.99            42.12                  0.8245        london               0.8922
       100             368.92            81.16                  0.7623      helsinki               0.8647
       300             914.07             3.07                  0.5461        munich               0.7239


In [12]:
# =========================================================
# GHI CHÍNH XÁC KẾT QUẢ THỰC TẾ CỦA BẠN VÀO RESULTS.CSV
# =========================================================

csv_rows = []

for item in results:
    d = item["Dimension"]
    
    # 1. Training Time
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "N/A",
        "metric_name": "training_time_sec",
        "metric_value": item["Training Time (s)"]
    })
    
    # 2. Model Size
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "N/A",
        "metric_name": "model_size_mb",
        "metric_value": item["Model Size (MB)"]
    })
    
    # 3. Cosine Similarity (doctor-physician)
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": f"{sim_pair[0]}-{sim_pair[1]}",
        "metric_name": "cosine_similarity",
        "metric_value": item[f"Sim ({sim_pair[0]}-{sim_pair[1]})"]
    })
    
    # 4. Analogy Result
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "germany-france+paris",
        "metric_name": "analogy_top1",
        "metric_value": item["Analogy Top-1"]
    })

    # 5. Downstream Task Sim
    csv_rows.append({
        "experiment": "exp4_dimension",
        "parameter_name": "vector_size",
        "parameter_value": d,
        "word_pair": "doc1-doc2",
        "metric_name": "downstream_doc_sim",
        "metric_value": item["Downstream Task Sim"]
    })

# Tạo DataFrame và ghi đè vào results.csv
df_final = pd.DataFrame(csv_rows)
df_final.to_csv("results.csv", index=False, encoding="utf-8")
print("\n[OK] Đã cập nhật chuẩn xác dữ liệu chạy thực tế vào 'results.csv'!")


[OK] Đã cập nhật chuẩn xác dữ liệu chạy thực tế vào 'results.csv'!


# 21. Evaluation — Word similarity

In [9]:
import pandas as pd
import numpy as np

word_pairs = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana")
]

human_intuition = {
    ("doctor", "physician"): 0.95,  
    ("car", "automobile"): 0.95,    
    ("king", "queen"): 0.80,       
    ("cat", "dog"): 0.70,        
    ("computer", "banana"): 0.05   
}

eval_data = []
for w1, w2 in word_pairs:
    if w1 in w2v_model.wv and w2 in w2v_model.wv:
        model_score = float(w2v_model.wv.similarity(w1, w2))
    else:
        model_score = 0.0
    
    intuition_score = human_intuition[(w1, w2)]
    eval_data.append({
        "Word Pair": f"{w1} – {w2}",
        "Model Similarity": round(model_score, 4),
        "Human Intuition": intuition_score
    })

df_eval = pd.DataFrame(eval_data)
df_eval["Model Rank"] = df_eval["Model Similarity"].rank(ascending=False, method="min").astype(int)
df_eval["Human Rank"] = df_eval["Human Intuition"].rank(ascending=False, method="min").astype(int)

df_eval = df_eval.sort_values(by="Model Rank").reset_index(drop=True)

print("=== BẢNG XẾP HẠNG WORD SIMILARITY ===")
print(df_eval.to_string(index=False))

from scipy.stats import spearmanr
corr, p_value = spearmanr(df_eval["Model Similarity"], df_eval["Human Intuition"])
print(f"\n--- ĐÁNH GIÁ ĐỊNH LƯỢNG (QUANTITATIVE METRIC) ---")
print(f"Hệ số tương quan thứ hạng Spearman (Spearman Correlation): {corr:.4f}")
print(f"p-value: {p_value:.4f}")

=== BẢNG XẾP HẠNG WORD SIMILARITY ===
         Word Pair  Model Similarity  Human Intuition  Model Rank  Human Rank
         cat – dog            0.7619             0.70           1           4
  car – automobile            0.7403             0.95           2           1
doctor – physician            0.7353             0.95           3           1
      king – queen            0.7020             0.80           4           3
 computer – banana            0.0451             0.05           5           5

--- ĐÁNH GIÁ ĐỊNH LƯỢNG (QUANTITATIVE METRIC) ---
Hệ số tương quan thứ hạng Spearman (Spearman Correlation): 0.3591
p-value: 0.5528


# 24. Application - Semantic Search

In [10]:
import numpy as np
import pandas as pd


def expand_query(query, model, topn=3):
    query_tokens = clean_and_tokenize(query)
    expanded_words = set(query_tokens)
    
    for token in query_tokens:
        if token in model.wv:
            similar_words = [word for word, score in model.wv.most_similar(token, topn=topn)]
            expanded_words.update(similar_words)
            
    return list(expanded_words)

def search_by_expansion(query, corpus_docs, model, top_k=3):
    expanded_words = expand_query(query, model, topn=3)
    print(f"Query ban đầu: '{query}'")
    print(f"Query được mở rộng (Expanded Words): {expanded_words}\n")
    
    doc_scores = []
    for idx, doc in enumerate(corpus_docs):
        score = sum(1 for token in doc if token in expanded_words)
        if score > 0:
            doc_scores.append((idx, score, doc))
    
    doc_scores.sort(key=lambda x: x[1], reverse=True)
    return doc_scores[:top_k]


def get_sentence_vector(tokens, model, dim=100):
    
    valid_vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(valid_vecs) == 0:
        return np.zeros(dim)
    return np.mean(valid_vecs, axis=0)

sample_corpus = corpus[:1000]
doc_vectors = np.array([get_sentence_vector(doc, w2v_model, dim=100) for doc in sample_corpus])

def search_dense(query, corpus_docs, doc_vectors, model, top_k=3):
    query_tokens = clean_and_tokenize(query)
    query_vec = get_sentence_vector(query_tokens, model, dim=100)
   
    norm_query = np.linalg.norm(query_vec)
    norm_docs = np.linalg.norm(doc_vectors, axis=1)
   
    norm_docs[norm_docs == 0] = 1e-10 
    
    cosine_sims = np.dot(doc_vectors, query_vec) / (norm_docs * norm_query)
    
    top_indices = np.argsort(cosine_sims)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append((idx, cosine_sims[idx], " ".join(corpus_docs[idx][:40]) + "..."))
    return results

test_query = "medical treatment"

print("=== TÌM KIẾM THEO CÁCH 1: QUERY EXPANSION ===")
results_exp = search_by_expansion(test_query, sample_corpus, w2v_model, top_k=3)
for rank, (doc_idx, score, doc_tokens) in enumerate(results_exp, 1):
    print(f"Top {rank} [Score: {score}] (Doc #{doc_idx}):")
    print(f"  '{" ".join(doc_tokens[:30])}...'\n")

print("="*60 + "\n")

print("=== TÌM KIẾM THEO CÁCH 2: DENSE EMBEDDING VECTOR ===")
results_dense = search_dense(test_query, sample_corpus, doc_vectors, w2v_model, top_k=3)
for rank, (doc_idx, score, doc_text) in enumerate(results_dense, 1):
    print(f"Top {rank} [Cosine Sim: {score:.4f}] (Doc #{doc_idx}):")
    print(f"  '{doc_text}'\n")

=== TÌM KIẾM THEO CÁCH 1: QUERY EXPANSION ===
Query ban đầu: 'medical treatment'
Query được mở rộng (Expanded Words): ['pathology', 'medical', 'treatments', 'nonsurgical', 'kybella', 'physicians', 'treatment', 'hospitalist']

Top 1 [Score: 21] (Doc #190):
  'this title is a comprehensive account of the key aspects of medical leadership a highly accessible text bookstyle resource it explores how the medical profession has evolved in tandem with...'

Top 2 [Score: 19] (Doc #282):
  'for the month of april only receive dermapen face neck treatment an alginate mask ionzyme machine treatment for only r all inclusive we are proud to launch the new dermapen...'

Top 3 [Score: 8] (Doc #709):
  'cosmetic surgery has become increasingly popular in the uk over the last ten years social media and cosmeticallyenhanced reality tv stars on shows such as made in chelsea and the...'


=== TÌM KIẾM THEO CÁCH 2: DENSE EMBEDDING VECTOR ===
Top 1 [Cosine Sim: 0.8234] (Doc #280):
  'a phase ii pilot study t